# Experiment 05 · Multi-Agent Games

**WalkingLab × Hands-On Modern RL companion experiment notebook**

Train a shared PPO policy in PettingZoo and replay all agents in one synchronized result.

- Resource profile: **CPU**
- Quick run in this notebook: **2,000** training units
- Full experiment: 20,000+ environment steps for cooperative navigation
- [Live ModelScope Studio](https://modelscope.cn/studios/walkinglab/hands-on-modern-rl-experiment05-multiagent-games)
- [Experiment source](https://github.com/walkinglabs/hands-on-modern-rl/tree/main/modelscope-space/hands-on-modern-rl-experiment05-multiagent-games)
- [Hands-On Modern RL](https://github.com/walkinglabs/hands-on-modern-rl) · [WalkingLab](https://modelscope.cn/organization/walkinglab)

The notebook imports the exact runtime used by the Studio. Change the parameters below, run the cells in order,
and compare the checkpoint curve with the final policy GIF or result image. The first setup can take longer because
native environments and simulator assets are cached; later runs reuse `/mnt/workspace/hands-on-modern-rl-notebooks`.


## 1. Question and run boundary

This experiment asks whether the selected policy improves on the task's evaluation metric as its training budget
increases. Start with the quick budget to verify the environment and logs. Then increase the budget only after the
complete result cell produces a curve and an artifact.

A normal ModelScope **CPU Notebook** is sufficient; no GPU is required.

A short smoke run proves that the pipeline executes; it does not prove convergence. Use the full budget above when
comparing algorithms or reporting a learned behavior.


## 2. Prepare the matching Studio runtime


In [ ]:
from __future__ import annotations

import hashlib
import os
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/walkinglabs/hands-on-modern-rl.git"
SPACE_SLUG = "hands-on-modern-rl-experiment05-multiagent-games"
INSTALL_DEPENDENCIES = True

def locate_or_clone_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "modelscope-space" / SPACE_SLUG).is_dir():
            return candidate
    workspace = Path("/mnt/workspace") if Path("/mnt/workspace").is_dir() else Path.cwd()
    target = workspace / "hands-on-modern-rl-notebooks" / "source"
    target.parent.mkdir(parents=True, exist_ok=True)
    if (target / ".git").is_dir():
        subprocess.run(["git", "-C", str(target), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(target)], check=True)
    return target

REPO_ROOT = locate_or_clone_repo()
SPACE_DIR = REPO_ROOT / "modelscope-space" / SPACE_SLUG
requirements = SPACE_DIR / "requirements.txt"
packages = SPACE_DIR / "packages.txt"
cache_root = Path("/mnt/workspace/hands-on-modern-rl-notebooks") if Path("/mnt/workspace").is_dir() else REPO_ROOT / ".cache" / "online-experiments"
cache_root.mkdir(parents=True, exist_ok=True)
digest = hashlib.sha256(requirements.read_bytes() + (packages.read_bytes() if packages.exists() else b"")).hexdigest()[:12]
marker = cache_root / f"{SPACE_SLUG}-{digest}.ready"

if INSTALL_DEPENDENCIES and not marker.exists():
    if packages.exists() and sys.platform.startswith("linux") and hasattr(os, "geteuid") and os.geteuid() == 0:
        system_packages = [line.strip() for line in packages.read_text().splitlines() if line.strip() and not line.startswith("#")]
        subprocess.run(["apt-get", "update"], check=True)
        subprocess.run(["apt-get", "install", "-y", "--no-install-recommends", *system_packages], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", str(requirements)], check=True)
    marker.touch()
else:
    print(f"Dependency cache ready: {marker}")

os.chdir(SPACE_DIR)
if str(SPACE_DIR) not in sys.path:
    sys.path.insert(0, str(SPACE_DIR))
print(f"Repository: {REPO_ROOT}")
print(f"Experiment runtime: {SPACE_DIR}")


## 3. Choose a task and train


In [ ]:
import importlib

if SPACE_SLUG.endswith("experiment10-minestudio"):
    subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--no-deps", "minestudio==1.1.6"], check=True)
if SPACE_SLUG.endswith("experiment11-unity-mlagents"):
    from bootstrap_mlagents import ensure_mlagents
    ensure_mlagents()

runtime = importlib.import_module("space_runtime")
tasks = {item["key"]: item for item in runtime.TASKS}
print("Available tasks:")
for key, item in tasks.items():
    title = item.get("title", {})
    print(f"  {key:20s} {title.get('en', title)} · {item.get('environment', 'environment provided by runtime')}")

TASK_KEY = "simple-spread"
TRAINING_BUDGET = 2000
LEARNING_RATE = 3e-4
GAMMA = 0.99
EPSILON = 0.10
SEED = 42

if TASK_KEY not in tasks:
    raise ValueError(f"Unknown TASK_KEY={TASK_KEY!r}. Choose one of {list(tasks)}")
selected_task = tasks[TASK_KEY]
print("\nSelected:", selected_task.get("title", {}).get("en", TASK_KEY))
print("Algorithm:", selected_task.get("algorithm"))
print("Budget:", TRAINING_BUDGET)


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt

print('Device: CPU')

print("Starting the same training generator used by the live Studio...\n")
events = []
for event in runtime.run(TASK_KEY, TRAINING_BUDGET, LEARNING_RATE, GAMMA, EPSILON, SEED):
    events.append(dict(event))
    message = event.get("log") or event.get("detail")
    if message:
        print(message, flush=True)

if not events:
    raise RuntimeError("The runtime returned no training events")
final_event = events[-1]
print("\nFinal phase:", final_event.get("phase", "complete"))
print("Final score:", final_event.get("score", "reported in the log"))


In [ ]:
from IPython.display import Image as NotebookImage

x = final_event.get("x", [])
y = final_event.get("y", [])
if x and y:
    fig, ax = plt.subplots(figsize=(8, 4.2))
    ax.plot(x, y, marker="o", color="#5b5ce2", linewidth=2)
    ax.set_title(f"{TASK_KEY} · checkpoint evaluation")
    ax.set_xlabel("Training progress")
    ax.set_ylabel("Evaluation score")
    ax.grid(alpha=0.25)
    plt.show()
else:
    print("This task reports its result through the artifact rather than a scalar learning curve.")

preview = final_event.get("preview")
if preview and Path(preview).exists():
    print("Learned-policy artifact:", preview)
    display(NotebookImage(filename=str(preview)))
else:
    print("No replay path was returned. Inspect the final log and the artifacts directory:", SPACE_DIR / "artifacts")

artifact = final_event.get("artifact") or final_event.get("model")
if artifact:
    print("Downloadable artifact:", artifact)


## 4. Read the result before increasing the budget

Compare the first and last checkpoint values, then inspect the replay. A rising curve with an implausible replay can
indicate reward shaping, evaluation, or rendering problems. A flat quick run is also inconclusive: this notebook's
default budget is a pipeline check. For a training claim, rerun with **20,000+ environment steps for cooperative navigation**, keep the seed fixed,
and compare at least three seeds before drawing a conclusion.
